# 06 — Whale Optimization Algorithm (WOA) Feature Selection

Binary WOA searches for a compact feature mask. This notebook uses the shared experiment pipeline so all optimizers receive the same data splits, classifiers, seeds, and evaluation rules.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Whale Optimization Algorithm

In [2]:
# -------------------------------------------------
# Binary Whale Optimization Algorithm (BWOA)
# -------------------------------------------------

import numpy as np


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def run_bwoa(
    obj_func,
    n_features,
    pop_size=30,
    iterations=50,
):

    # Initialize whales
    positions = np.random.randint(0, 2, (pop_size, n_features))

    fitness = np.array([obj_func(w) for w in positions])

    best_idx = np.argmin(fitness)
    best_position = positions[best_idx].copy()
    best_score = fitness[best_idx]

    convergence = []

    b = 1

    for t in range(iterations):

        a = 2 - 2 * (t / iterations)

        for i in range(pop_size):

            r1 = np.random.rand()
            r2 = np.random.rand()

            A = 2 * a * r1 - a
            C = 2 * r2

            p = np.random.rand()

            new_position = np.zeros(n_features)

            if p < 0.5:

                if abs(A) < 1:

                    D = np.abs(C * best_position - positions[i])
                    X = best_position - A * D

                else:

                    rand_idx = np.random.randint(pop_size)
                    rand_whale = positions[rand_idx]

                    D = np.abs(C * rand_whale - positions[i])
                    X = rand_whale - A * D

            else:

                l = np.random.uniform(-1, 1)

                D = np.abs(best_position - positions[i])

                X = (
                    D
                    * np.exp(b * l)
                    * np.cos(2 * np.pi * l)
                    + best_position
                )

            probs = sigmoid(X)

            new_position = (
                np.random.rand(n_features) < probs
            ).astype(int)

            # Prevent empty subset
            if new_position.sum() == 0:
                new_position[np.random.randint(n_features)] = 1

            positions[i] = new_position

        fitness = np.array([obj_func(w) for w in positions])

        idx = np.argmin(fitness)

        if fitness[idx] < best_score:
            best_score = fitness[idx]
            best_position = positions[idx].copy()

        convergence.append(best_score)

    return best_position, best_score, convergence

## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [3]:
from utils.experiments import run_feature_selector


def woa_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_bwoa(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        iterations=iterations,
    )


woa_results = run_feature_selector("WOA", woa_runner)
woa_results.tail()


Saved: ('breast', 'svm', 0)


Saved: ('breast', 'random_forest', 0)


Saved: ('breast', 'xgboost', 0)


Saved: ('heart', 'svm', 0)


Saved: ('heart', 'random_forest', 0)


Saved: ('heart', 'xgboost', 0)


,Dataset,Classifier,Algorithm,Seed,ValidationFitness,Accuracy,Precision,Recall,F1,ROC_AUC,Features,SelectionRuntime,TestRuntime,SelectedFeatureNames,MaskFile,ConvergenceFile
1,breast,random_forest,WOA,0,0.031053,0.982456,1.000000,0.952381,0.975610,0.998347,15,5.420512,0.263470,"[""radius_mean"", ""perimeter_mean"", ""smoothness_...",results/smoke/artifacts/breast__random_forest_...,results/smoke/artifacts/breast__random_forest_...
2,breast,xgboost,WOA,0,0.022035,0.938596,0.948718,0.880952,0.913580,0.978505,14,2.380585,0.083866,"[""concavity_mean"", ""concave points_mean"", ""fra...",results/smoke/artifacts/breast__xgboost__woa__...,results/smoke/artifacts/breast__xgboost__woa__...
3,heart,svm,WOA,0,0.145491,0.798913,0.842105,0.784314,0.812183,0.889048,14,0.455233,0.020781,"[""trestbps"", ""chol"", ""thalch"", ""ca"", ""sex_Fema...",results/smoke/artifacts/heart__svm__woa__seed0...,results/smoke/artifacts/heart__svm__woa__seed0...
4,heart,random_forest,WOA,0,0.150072,0.788043,0.831579,0.774510,0.802030,0.855093,12,6.602225,0.316469,"[""age"", ""trestbps"", ""thalch"", ""oldpeak"", ""sex_...",results/smoke/artifacts/heart__random_forest__...,results/smoke/artifacts/heart__random_forest__...
5,heart,xgboost,WOA,0,0.151672,0.809783,0.825243,0.833333,0.829268,0.870038,16,1.987532,0.083378,"[""age"", ""trestbps"", ""chol"", ""ca"", ""sex_Female""...",results/smoke/artifacts/heart__xgboost__woa__s...,results/smoke/artifacts/heart__xgboost__woa__s...
